In [11]:
import time
import re
from typing import List, Dict, Any, Optional

import pandas as pd
from Bio import Entrez


# -----------------------------
# Configuration
# -----------------------------
Entrez.email = "your_email@example.com"   # required by NCBI


BIOPROJECT_IDS = [
    "PRJNA360672",
    "PRJNA524291",
    "PRJNA575811",
    "PRJNA576556",
    "PRJNA630329",
    "PRJNA630325",
    "PRJNA650220",
    "PRJNA907389",
    "PRJNA954555",
    "PRJNA1048100",
    "PRJNA1165229",
    "PRJNA988282",
    "PRJNA951783",
    "PRJNA665377",
    "PRJNA325476",
    "PRJNA497709",
    "PRJEB101302",
    "PRJNA1336731",
    "PRJNA1297037",
    "PRJNA1254620",
    "PRJNA1089063",
    "PRJNA592182",
    "PRJNA588472",
    "PRJNA576764",
    "PRJNA576020",
]


# -----------------------------
# Helper functions
# -----------------------------
def chunked(items: List[str], size: int) -> List[List[str]]:
    for i in range(0, len(items), size):
        yield items[i:i + size]


def safe_get(d: Any, *keys, default=None):
    """Safely walk nested dicts/lists."""
    cur = d
    for key in keys:
        try:
            if isinstance(cur, list) and isinstance(key, int):
                cur = cur[key]
            else:
                cur = cur[key]
        except (KeyError, IndexError, TypeError):
            return default
    return cur


def extract_from_description(description: str, pattern: str) -> Optional[str]:
    if not description:
        return None
    m = re.search(pattern, description, flags=re.IGNORECASE)
    return m.group(1).strip() if m else None


def ncbi_retry(func, *args, retries: int = 3, sleep_sec: float = 0.5, **kwargs):
    last_err = None
    for attempt in range(retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            last_err = e
            time.sleep(sleep_sec * (attempt + 1))
    raise last_err


def bioproject_accession_to_uid(accession: str) -> Optional[str]:
    """Convert PRJNA/PRJEB accession to NCBI BioProject UID."""
    with ncbi_retry(
        Entrez.esearch,
        db="bioproject",
        term=f"{accession}[Project Accession]",
        retmode="xml"
    ) as handle:
        record = Entrez.read(handle)

    id_list = record.get("IdList", [])
    return id_list[0] if id_list else None


def fetch_bioproject_summary(uid: str) -> Dict[str, Any]:
    """Fetch BioProject summary from NCBI."""
    with ncbi_retry(
        Entrez.esummary,
        db="bioproject",
        id=uid,
        retmode="xml"
    ) as handle:
        summary = Entrez.read(handle)

    docs = summary.get("DocumentSummarySet", {}).get("DocumentSummary", [])
    return docs[0] if docs else {}


def elink_ids(dbfrom: str, db: str, uid: str) -> List[str]:
    """Get linked record IDs from one NCBI db to another."""
    with ncbi_retry(
        Entrez.elink,
        dbfrom=dbfrom,
        db=db,
        id=uid,
        retmode="xml"
    ) as handle:
        record = Entrez.read(handle)

    ids = []
    for linkset in record:
        for dbset in linkset.get("LinkSetDb", []):
            for link in dbset.get("Link", []):
                link_id = link.get("Id")
                if link_id:
                    ids.append(str(link_id))
    return ids


def parse_bioproject_summary(accession: str, uid: str, summary: Dict[str, Any]) -> Dict[str, Any]:
    """
    Parse the somewhat inconsistent BioProject esummary record.
    NCBI fields can vary, so we collect what is consistently useful.
    """
    title = summary.get("Project_Title")
    description = summary.get("Project_Description")
    name = summary.get("Project_Name")
    status = summary.get("Project_Status")
    submission_date = summary.get("Submission_Date")
    last_update = summary.get("Last_Update")
    organism = summary.get("Organism_Name")
    taxonomy_id = summary.get("Organism_Id")
    target_scope = summary.get("Project_TargetScope")
    data_type = summary.get("Project_Data_Type")
    relevance = summary.get("Project_Relevance")
    locus_tag_prefix = summary.get("LocusTagPrefix")
    submitter = summary.get('Submitter_Organization')
    registration_date = summary.get("Registration_Date"
                                    )
    # Some records hide useful text in the description
    center = summary.get("Submission_Center") or extract_from_description(description, r"submitter:\s*([^;|]+)")
    umbrella = summary.get("Project_Umbrella") or None

    row = {
        "bioproject_accession": accession,
        "bioproject_uid": uid,
        "project_name": name,
        "project_title": title,
        "description": description,
        "organism": organism,
        "taxonomy_id": taxonomy_id,
        "target_scope": target_scope,
        "data_type": data_type,
        "relevance": relevance,
        "status": status,
        "registration_date": registration_date,
        "submission_date": submission_date,
        "last_update": last_update,
        "submission_center": center,
        "submitter": submitter,
        "umbrella_project": umbrella,
        "locus_tag_prefix": locus_tag_prefix,
        "bioproject_url": f"https://www.ncbi.nlm.nih.gov/bioproject/{accession}",
        "sra_search_url": f"https://www.ncbi.nlm.nih.gov/sra?linkname=bioproject_sra_all&from_uid={uid}",
        "biosample_search_url": f"https://www.ncbi.nlm.nih.gov/biosample?linkname=bioproject_biosample_all&from_uid={uid}",
        "assembly_search_url": f"https://www.ncbi.nlm.nih.gov/assembly?linkname=bioproject_assembly_all&from_uid={uid}",
        "pubmed_search_url": f"https://pubmed.ncbi.nlm.nih.gov/?term={accession}",
    }
    return row


def collect_linked_resources(uid: str) -> Dict[str, Any]:
    """Collect linked SRA/BioSample/Assembly/PubMed IDs."""
    sra_ids = elink_ids("bioproject", "sra", uid)
    biosample_ids = elink_ids("bioproject", "biosample", uid)
    assembly_ids = elink_ids("bioproject", "assembly", uid)
    pubmed_ids = elink_ids("bioproject", "pubmed", uid)

    return {
        "n_sra_records": len(sra_ids),
        "sra_ids": ",".join(sra_ids) if sra_ids else None,
        "n_biosample_records": len(biosample_ids),
        "biosample_ids": ",".join(biosample_ids) if biosample_ids else None,
        "n_assembly_records": len(assembly_ids),
        "assembly_ids": ",".join(assembly_ids) if assembly_ids else None,
        "n_pubmed_records": len(pubmed_ids),
        "pubmed_ids": ",".join(pubmed_ids) if pubmed_ids else None,
    }


def fetch_bioproject_table(accessions: List[str], add_linked_resources: bool = True, delay: float = 0.34) -> pd.DataFrame:
    rows = []

    for accession in accessions:
        try:
            uid = bioproject_accession_to_uid(accession)
            if not uid:
                rows.append({
                    "bioproject_accession": accession,
                    "error": "BioProject accession not found in NCBI"
                })
                continue

            summary = fetch_bioproject_summary(uid)
            row = parse_bioproject_summary(accession, uid, summary)

            if add_linked_resources:
                linked = collect_linked_resources(uid)
                row.update(linked)

            rows.append(row)

            # Respect NCBI rate limits
            time.sleep(delay)

        except Exception as e:
            rows.append({
                "bioproject_accession": accession,
                "error": str(e),
            })

    df = pd.DataFrame(rows)

    # Put the most useful columns first if they exist
    preferred_order = [
        "bioproject_accession",
        "bioproject_uid",
        "project_title",
        "project_name",
        "organism",
        "taxonomy_id",
        "data_type",
        "target_scope",
        "relevance",
        "status",
        "registration_date",
        "submission_date",
        "last_update",
        "submitter",
        "submission_center",
        "description",
        "n_sra_records",
        "n_biosample_records",
        "n_assembly_records",
        "n_pubmed_records",
        "bioproject_url",
        "sra_search_url",
        "biosample_search_url",
        "assembly_search_url",
        "pubmed_search_url",
        "sra_ids",
        "biosample_ids",
        "assembly_ids",
        "pubmed_ids",
        "umbrella_project",
        "locus_tag_prefix",
        "error",
    ]

    existing_cols = [c for c in preferred_order if c in df.columns]
    remaining_cols = [c for c in df.columns if c not in existing_cols]
    return df[existing_cols + remaining_cols]

In [ ]:
# -----------------------------
# Run
# -----------------------------
if __name__ == "__main__":
    df = fetch_bioproject_table(BIOPROJECT_IDS, add_linked_resources=True)

    print(df.head())
    df.to_csv("bioproject_metadata.csv", index=False)

In [4]:
projects = """PRJEB101302
PRJNA1336731
PRJNA1297037
PRJNA1254620
PRJNA1089063
PRJNA592182
PRJNA588472
PRJNA576764
PRJNA576020""".split("\n")

In [5]:
df[df["bioproject_accession"].isin(projects)]

,bioproject_accession,bioproject_uid,project_title,project_name,organism,taxonomy_id,data_type,target_scope,relevance,status,...,sra_search_url,biosample_search_url,assembly_search_url,pubmed_search_url,sra_ids,biosample_ids,assembly_ids,pubmed_ids,umbrella_project,locus_tag_prefix
16,PRJEB101302,1364369,Exaiptasia longitudinal microbiomes across tem...,Exaiptasia temperature across time,,None,Other,None,None,None,...,https://www.ncbi.nlm.nih.gov/sra?linkname=biop...,https://www.ncbi.nlm.nih.gov/biosample?linknam...,https://www.ncbi.nlm.nih.gov/assembly?linkname...,https://pubmed.ncbi.nlm.nih.gov/?term=PRJEB101302,"41494763,41494758,41494757,41494756,41494755,4...","53258304,53258289,53258279,53258278,53258275,5...",None,None,None,None
17,PRJNA1336731,1336731,Symbiotic state modulates microbiome recovery ...,,,None,Raw sequence reads,None,None,None,...,https://www.ncbi.nlm.nih.gov/sra?linkname=biop...,https://www.ncbi.nlm.nih.gov/biosample?linknam...,https://www.ncbi.nlm.nih.gov/assembly?linkname...,https://pubmed.ncbi.nlm.nih.gov/?term=PRJNA133...,"41390731,41390730,41390729,41390728,41390727,4...","53094953,53094952,53094951,53094950,53094949,5...",None,None,None,None
18,PRJNA1297037,1297037,Exaiptasia diaphana Aiptasia strains F003 H2 m...,Exaiptasia diaphana Aiptasia strains F003 H2 m...,coral metagenome,None,Raw sequence reads,None,None,None,...,https://www.ncbi.nlm.nih.gov/sra?linkname=biop...,https://www.ncbi.nlm.nih.gov/biosample?linknam...,https://www.ncbi.nlm.nih.gov/assembly?linkname...,https://pubmed.ncbi.nlm.nih.gov/?term=PRJNA129...,None,None,None,None,None,None
19,PRJNA1254620,1254620,Manipulation of the Symbiodiniaceae microbiome...,,,None,Raw sequence reads,None,None,None,...,https://www.ncbi.nlm.nih.gov/sra?linkname=biop...,https://www.ncbi.nlm.nih.gov/biosample?linknam...,https://www.ncbi.nlm.nih.gov/assembly?linkname...,https://pubmed.ncbi.nlm.nih.gov/?term=PRJNA125...,"38312553,38312552,38312551,38312550,38312549,3...","48115599,48115598,48115597,48115596,48115595,4...",None,None,None,None
20,PRJNA1089063,1089063,Exaiptasia diaphana isolate:F003 Genome sequen...,Exaiptasia diaphana isolate:F003,Exaiptasia diaphana,None,Genome sequencing,None,None,None,...,https://www.ncbi.nlm.nih.gov/sra?linkname=biop...,https://www.ncbi.nlm.nih.gov/biosample?linknam...,https://www.ncbi.nlm.nih.gov/assembly?linkname...,https://pubmed.ncbi.nlm.nih.gov/?term=PRJNA108...,"37056262,32296712,37056262,32296712","46427288,40517710,46427288,40517710,46427288,4...","32329211,32329211",None,None,None
21,PRJNA592182,592182,Comparison of gene expression in Exaiptasia pa...,Comparison of gene expression in Exaiptasia pa...,Exaiptasia diaphana,None,Transcriptome or Gene expression,None,None,None,...,https://www.ncbi.nlm.nih.gov/sra?linkname=biop...,https://www.ncbi.nlm.nih.gov/biosample?linknam...,https://www.ncbi.nlm.nih.gov/assembly?linkname...,https://pubmed.ncbi.nlm.nih.gov/?term=PRJNA592182,"9495150,9495149,9495148,9495147,9495146,949514...","13413534,13413533,13413532,13413531,13413530,1...",None,33927382,None,None
22,PRJNA588472,588472,Comparison of gene expression in symbiotic and...,Comparison of gene expression in symbiotic and...,Exaiptasia diaphana,None,Transcriptome or Gene expression,None,None,None,...,https://www.ncbi.nlm.nih.gov/sra?linkname=biop...,https://www.ncbi.nlm.nih.gov/biosample?linknam...,https://www.ncbi.nlm.nih.gov/assembly?linkname...,https://pubmed.ncbi.nlm.nih.gov/?term=PRJNA588472,"9360141,9360140,9360139,9360138,9360137,936013...","13245678,13245677,13245676,13245675,13245674,1...",None,33927382,None,None
23,PRJNA576764,576764,Thermal stress response of GBR Exaiptasia's mi...,,,None,Raw sequence reads,None,None,None,...,https://www.ncbi.nlm.nih.gov/sra?linkname=biop...,https://www.ncbi.nlm.nih.gov/biosample?linknam...,https://www.ncbi.nlm.nih.gov/assembly?linkname...,https://pubmed.ncbi.nlm.nih.gov/?term=PRJNA576764,"9174133,9174132,9174131,9174130,9174129,917412...","13006828,13006827,13006826,13006825,

In [12]:
uid = bioproject_accession_to_uid("PRJNA1336731")
summary = fetch_bioproject_summary(uid)
row = parse_bioproject_summary("PRJNA1336731", uid, summary)

In [13]:
summary

DictElement({'TaxId': '0', 'Project_Id': '1336731', 'Project_Acc': 'PRJNA1336731', 'Project_Type': 'Primary submission', 'Project_Data_Type': 'Raw sequence reads', 'Sort_By_ProjectType': '99631', 'Sort_By_DataType': '67402', 'Sort_By_Organism': '763371', 'Project_Subtype': '', 'Project_Target_Scope': 'Multispecies', 'Project_Target_Material': 'Genome', 'Project_Target_Capture': 'Whole', 'Project_MethodType': 'Sequencing', 'Project_Method': '', 'Project_Objectives_List': [{'Project_ObjectivesType': 'Raw Sequence Reads', 'Project_Objectives': ''}], 'Registration_Date': '2025/09/30 00:00', 'Project_Name': '', 'Project_Title': 'Symbiotic state modulates microbiome recovery following depletion in a facultatively symbiotic cnidarian', 'Project_Description': 'This study explored microbiome depletion and recovery following antibiotic treatment in symbiotic and aposymbiotic Exaiptasia pallida, a facultatively symbiotic sea anemone. We used 16S gene sequencing of the bacterial microbiome to char

In [8]:
row

{'bioproject_accession': 'PRJNA1336731',
 'bioproject_uid': '1336731',
 'project_name': '',
 'project_title': 'Symbiotic state modulates microbiome recovery following depletion in a facultatively symbiotic cnidarian',
 'description': 'This study explored microbiome depletion and recovery following antibiotic treatment in symbiotic and aposymbiotic Exaiptasia pallida, a facultatively symbiotic sea anemone. We used 16S gene sequencing of the bacterial microbiome to characterize taxa during recovery following antibiotic treatment. We characterized the host molecular response to microbiome depletion and recovery using whole-animal gene expression profiling and Western blotting of the innate immune transcription factor NF-kappaB. This study reveals dynamics of microbiome recovery following depletion and the influence of microbiome community members on host gene expression, providing a foundation for work for future research involving pairwise interactions between beneficial microorganisms a

In [ ]:
for accession in accessions:
        try:
            uid = bioproject_accession_to_uid(accession)
            if not uid:
                rows.append({
                    "bioproject_accession": accession,
                    "error": "BioProject accession not found in NCBI"
                })
                continue

            summary = fetch_bioproject_summary(uid)
            row = parse_bioproject_summary(accession, uid, summary)

            if add_linked_resources:
                linked = collect_linked_resources(uid)
                row.update(linked)

            rows.append(row)

            # Respect NCBI rate limits
            time.sleep(delay)

        except Exception as e:
            rows.append({
                "bioproject_accession": accession,
                "error": str(e),
            })
